In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "pdf.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 110})

ROI = "HIP"
CHEMS = None
SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

df = pd.read_parquet(f"{SLIM_DIR}/{ROI}_slim.parquet")
if CHEMS:
    df = df[df["chem"].isin(CHEMS)]
genes = [g for g in GRMS if g in df.columns]
cls_num = pd.to_numeric(df["class"].astype(str).str[:2], errors="coerce").fillna(0)
df["neuron"] = ~cls_num.between(30, 34)
os.makedirs(FIG_DIR, exist_ok=True)
tag = ROI + ("_" + "-".join(c.replace("WMB-10X", "") for c in CHEMS) if CHEMS else "")
print(ROI, len(df), "cells;", df["subclass"].nunique(), "subclasses; chem:", df["chem"].value_counts().to_dict())

d = df.dropna(subset=["allen_x", "allen_y"])
if len(d) > 150000:
    d = d.sample(150000, random_state=0)
classes = sorted(d["class"].unique())
cmap = plt.get_cmap("tab20")
col = {c: cmap(i % 20) for i, c in enumerate(classes)}
fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(d["allen_x"], d["allen_y"], s=1.5, c=[col[c] for c in d["class"]], linewidths=0, rasterized=True)
cent = d.groupby("subclass")[["allen_x", "allen_y"]].median()
big = d["subclass"].value_counts().index[:35]
for sc in big:
    ax.text(cent.loc[sc, "allen_x"], cent.loc[sc, "allen_y"], sc, fontsize=6, ha="center", va="center")
handles = [plt.Line2D([], [], marker="o", ls="", color=col[c], label=c, markersize=5) for c in classes]
ax.legend(handles=handles, fontsize=6, frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5), title="class")
ax.set_xlabel("Allen UMAP 1")
ax.set_ylabel("Allen UMAP 2")
ax.set_title(f"{ROI}: {len(df):,} cells, colored by class, top 35 subclasses labeled", fontsize=10)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.savefig(f"{FIG_DIR}/{tag}_fig1_umap_subclass.png", dpi=300, bbox_inches="tight")
plt.show()